# Hourly Delivery Demand Forecasting (LaDe Dataset)

**Goal:** Predict how many deliveries will happen each hour in each delivery zone (AOI).

Each row in the final dataset represents one **hour** in one **AOI**, defined by:
- `city`, `region_id`, `aoi_id`, `hour`

**Target variable:** `demand_count` — the number of deliveries in that AOI during that hour.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# Pandas display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

print("Ready.")
print("Ready.")

Ready.
Ready.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Time grouping
TIME_BUCKET = "h"   # Hourly

# Split ratios based on time
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20
TEST_RATIO  = 0.20

# Lag/rolling configuration
LAGS         = [1, 2, 3, 6, 12, 24, 48, 168]   # expanded: sub-hour trends + daily + weekly
ROLL_WINDOWS = [6, 24, 72, 168]                  # 6h, 1d, 3d, 1wk

# Set DATE_START/DATE_END or MAX_AOIS to None to disable the filter.
DATE_START = None
DATE_END   = None
MAX_AOIS   = 4000

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_PATH   = "/content/drive/MyDrive/Senior Project/LaDe-Dataset/Dataset.csv"
OUTPUT_PATH = "/content/drive/MyDrive/Senior Project/LaDe-Dataset/lade_hourly_features.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
df_raw.head()

Shape: (1861600, 17)


,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds
0,583722,0,Hangzhou,175,120.17895,30.26401,749,1,10-30 09:20:00,10-30 09:20:00,120.20600,30.28657,10-30 10:30:00,10-30 10:30:00,120.17908,30.26392,1030
1,2819059,0,Hangzhou,175,120.17899,30.26336,749,1,10-31 09:47:00,10-31 09:47:00,120.20591,30.28651,10-31 10:40:00,10-31 10:40:00,120.17884,30.26360,1031
2,2879432,0,Hangzhou,175,120.17896,30.26404,749,1,10-22 10:11:00,10-22 10:11:00,120.20598,30.28668,10-22 15:03:00,10-22 15:03:00,120.17939,30.26395,1022
3,392295,0,Hangzhou,175,120.17897,30.26408,749,1,10-26 09:41:00,10-26 09:41:00,120.20600,30.28657,10-26 10:30:00,10-26 10:30:00,120.17925,30.26465,1026
4,231864,0,Hangzhou,175,120.17888,30.26406,749,1,10-31 15:58:00,10-31 15:58:00,120.20605,30.28666,10-31 16:41:00,10-31 16:41:00,120.17886,30.26402,1031


# Columns We Use

**Location columns** — define where each delivery happens:
- `city`, `region_id`, `aoi_id`, `aoi_type`

**Time columns:**
- `ds` — date in `MMDD` format (e.g., `0925` = September 25)
- `accept_time` — when the order was accepted (`MM-DD HH:MM:SS`). We prepend `2022` to get a full datetime.

**Target (derived):**
- `demand_count` — order count per `(city, region_id, aoi_id, hour)`, computed after hourly bucketing.

In [5]:
# ── Data Quality Checks ───────────────────────────────────────────────────────

key_cols = ["city", "region_id", "aoi_id", "aoi_type", "accept_time", "ds"]
print("Missing values in key columns:")
print(df_raw[key_cols].isna().sum())

print()
print("Unique cities:  ", df_raw["city"].nunique())
print("Unique regions: ", df_raw["region_id"].nunique())
print("Unique AOIs:    ", df_raw["aoi_id"].nunique())
print("Unique couriers:", df_raw["courier_id"].nunique())

print()
dup_orders = df_raw["order_id"].duplicated().sum()
print("Duplicate order_id count:", dup_orders)

Missing values in key columns:
city           0
region_id      0
aoi_id         0
aoi_type       0
accept_time    0
ds             0
dtype: int64

Unique cities:   1
Unique regions:  51
Unique AOIs:     17714
Unique couriers: 1392

Duplicate order_id count: 0


# Time Parsing

**Building a full datetime:** The raw `accept_time` is `MM-DD HH:MM:SS`. We prepend `2022` to get a proper timestamp:  
`"09-25 08:08:00"` → `"2022-09-25 08:08:00"`

**Hourly bucketing:** Each timestamp is rounded down to the hour:  
`2022-09-25 08:08:00` → `2022-09-25 08:00:00`

We use `accept_time` (when the order enters the system) to define demand, since it reflects incoming workload — useful for dispatch planning and courier allocation.

In [6]:
# ── Timestamp Parsing (prepend year to accept_time: MM-DD HH:MM:SS → YYYY-MM-DD HH:MM:SS) ─

DATA_YEAR = "2022"

df = df_raw.copy()

# accept_time format is "MM-DD HH:MM:SS" — prepend year to get full datetime
df["accept_dt"] = pd.to_datetime(
    DATA_YEAR + "-" + df["accept_time"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Unparsed accept_time rows:", df["accept_dt"].isna().sum())
df = df.dropna(subset=["accept_dt"]).copy()

# Optional date-range filtering
if DATE_START is not None:
    df = df[df["accept_dt"] >= DATE_START]
if DATE_END is not None:
    end_exclusive = pd.to_datetime(DATE_END) + pd.Timedelta(days=1)
    df = df[df["accept_dt"] < end_exclusive]

print("Time range after filtering:", df["accept_dt"].min(), "→", df["accept_dt"].max())
df[["ds", "accept_time", "accept_dt"]].head()

Unparsed accept_time rows: 0
Time range after filtering: 2022-05-01 07:11:00 → 2022-10-31 22:36:00


,ds,accept_time,accept_dt
0,1030,10-30 09:20:00,2022-10-30 09:20:00
1,1031,10-31 09:47:00,2022-10-31 09:47:00
2,1022,10-22 10:11:00,2022-10-22 10:11:00
3,1026,10-26 09:41:00,2022-10-26 09:41:00
4,1031,10-31 15:58:00,2022-10-31 15:58:00


# Hourly Demand Aggregation

We count orders per `(city, region_id, aoi_id, aoi_type, bucket_hour)` to get `demand_count` — the number of deliveries in each zone during each hour.

In [7]:
# ── Hourly Aggregation ────────────────────────────────────────────────────────

df["bucket_hour"] = df["accept_dt"].dt.floor(TIME_BUCKET)

demand_agg = (
    df.groupby(["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"], as_index=False)
      .agg(demand_count=("order_id", "count"))
      .sort_values(["city", "region_id", "aoi_id", "bucket_hour"])
)

print("Aggregated shape:", demand_agg.shape)

# ── Demand Distribution (sanity check) ───────────────────────────────────────
print("\nDemand count distribution (non-zero hours only):")
print("  Min:   ", demand_agg["demand_count"].min())
print("  Max:   ", demand_agg["demand_count"].max())
print("  Mean:  ", round(demand_agg["demand_count"].mean(), 3))
print("  Median:", demand_agg["demand_count"].median())
print()
print(demand_agg["demand_count"].quantile([0.25, 0.5, 0.75, 0.9, 0.99]))

demand_agg.head()

Aggregated shape: (907353, 6)

Demand count distribution (non-zero hours only):
  Min:    1
  Max:    58
  Mean:   2.052
  Median: 1.0

0.25     1.0
0.50     1.0
0.75     2.0
0.90     4.0
0.99    12.0
Name: demand_count, dtype: float64


,city,region_id,aoi_id,aoi_type,bucket_hour,demand_count
0,Hangzhou,0,749,1,2022-05-01 17:00:00,1
1,Hangzhou,0,749,1,2022-05-03 13:00:00,1
2,Hangzhou,0,749,1,2022-05-05 09:00:00,1
3,Hangzhou,0,749,1,2022-05-23 08:00:00,1
4,Hangzhou,0,749,1,2022-05-23 15:00:00,1


# Zero-Filling Missing Hours

**Problem:** The raw data only has rows where orders exist. Hours with no deliveries are missing entirely. Without them, the model never learns what "zero demand" looks like, leading to overestimated forecasts.

**Solution:** We build a complete hourly timeline for every AOI and fill missing hours with `demand_count = 0`.

To speed this up, we pre-group the data into a dictionary so each AOI lookup is O(1) instead of scanning the full table.

In [8]:
# ── AOI Selection ─────────────────────────────────────────────────────────────

if MAX_AOIS is not None:
    aoi_totals = (
        demand_agg.groupby(["city", "region_id", "aoi_id", "aoi_type"])["demand_count"]
        .sum()
        .sort_values(ascending=False)
        .head(MAX_AOIS)
        .reset_index()
    )
    aoi_keys = aoi_totals[["city", "region_id", "aoi_id", "aoi_type"]]
else:
    aoi_keys = demand_agg[["city", "region_id", "aoi_id", "aoi_type"]].drop_duplicates().reset_index(drop=True)

# Global hourly range
start_hour = demand_agg["bucket_hour"].min()
end_hour   = demand_agg["bucket_hour"].max()
full_hours = pd.date_range(start=start_hour, end=end_hour, freq=TIME_BUCKET)

print(f"AOIs selected:          {len(aoi_keys)}")
print(f"Hours per AOI:          {len(full_hours)}")
print(f"Expected total rows:    {len(aoi_keys) * len(full_hours):,}")

# ── Pre-group demand_agg for O(1) AOI lookup (key optimisation) ───────────────
# Instead of scanning all rows per AOI in the loop, we group once here.
agg_grouped = {
    key: grp[["bucket_hour", "demand_count"]].set_index("bucket_hour")
    for key, grp in demand_agg.groupby(["city", "region_id", "aoi_id", "aoi_type"])
}

print(f"\nPre-grouped AOI lookup table built: {len(agg_grouped)} entries")

AOIs selected:          4000
Hours per AOI:          4408
Expected total rows:    17,632,000

Pre-grouped AOI lookup table built: 19499 entries


In [9]:
# ── Grid Builder (zero-fill) ──────────────────────────────────────────────────

def build_aoi_grid(agg_grouped, key_row, full_hours):
    """
    Build a complete hourly time series for one AOI.
    Missing hours are filled with demand_count = 0.
    Uses O(1) dict lookup instead of scanning the full dataframe.
    """
    key = (key_row["city"], key_row["region_id"], key_row["aoi_id"], key_row["aoi_type"])

    g = agg_grouped.get(key, pd.DataFrame(columns=["demand_count"]))
    g = g.reindex(full_hours)
    g["demand_count"] = g["demand_count"].fillna(0).astype("int32")  # int32 avoids int16 overflow risk

    g = g.reset_index().rename(columns={"index": "bucket_hour"})
    g["city"]      = key_row["city"]
    g["region_id"] = key_row["region_id"]
    g["aoi_id"]    = key_row["aoi_id"]
    g["aoi_type"]  = key_row["aoi_type"]

    return g[["city", "region_id", "aoi_id", "aoi_type", "bucket_hour", "demand_count"]]

# Advanced Feature Engineering

The pipeline below generates **~60+ features** across six categories. Each feature is designed to teach the model *why* demand changes, not just *what* it was.

## Feature Categories

| Category | Features | What It Captures |
|---|---|---|
| **Cyclical Temporal** | `hour_sin/cos`, `dow_sin/cos`, `month_sin/cos` | Circular continuity (hour 23 ≈ hour 0). Trees can't learn this from raw integers. |
| **Lag Features** | `lag_1..lag_168` (8 lags) | Short-term memory (1-12h), daily pattern (24/48h), weekly seasonality (168h). |
| **Rolling Statistics** | `roll_{6,24,72,168}_{mean,std,min,max,range}` | Demand level, volatility, extremes, and spread at multiple timescales. |
| **Momentum & Trend** | `momentum_1_24`, `acceleration`, `lag1_over_roll24`, `trend_24_168` | Whether demand is rising/falling, spiking vs normal, short-term trend vs long-term baseline. |
| **Zero-Demand** | `is_nonzero`, `zero_ratio_24`, `consec_zeros` | Distinguishes true inactivity from low demand; captures dormancy streaks. |
| **Spatial / Neighbor** | `aoi_lng/lat`, `aoi_busyness`, `region_lag1_mean`, `region_lag24_mean` | AOI location, baseline activity level, and what neighboring AOIs are doing. |
| **Interaction** | `hour_sin×weekend`, `is_peak×weekend`, `peak_hour` (data-driven) | Demand behaves differently on weekends; peak hours shift by day type. |
| **Categorical** | `aoi_type` kept as-is; `aoi_id`/`region_id` NOT used as ordinal features | IDs replaced with meaningful encodings (coordinates, busyness). Target encoding deferred to training. |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ADVANCED FEATURE ENGINEERING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── STEP 1: Complete hourly grid ──────────────────────────────────────────────
print("Step 1: Building complete hourly grid...")

aoi_hours = aoi_keys.assign(_k=1).merge(
    pd.DataFrame({"bucket_hour": full_hours, "_k": 1}), on="_k"
).drop(columns="_k")

model_df = aoi_hours.merge(
    demand_agg[["city", "region_id", "aoi_id", "aoi_type", "bucket_hour", "demand_count"]],
    on=["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"],
    how="left"
)
model_df["demand_count"] = model_df["demand_count"].fillna(0).astype("int32")
model_df = model_df.sort_values(["city", "region_id", "aoi_id", "bucket_hour"]).reset_index(drop=True)
print(f"  Grid shape: {model_df.shape}")

# ── STEP 2: Calendar & Cyclical Temporal Features ─────────────────────────────
print("Step 2: Calendar & cyclical features...")

model_df["hour"]       = model_df["bucket_hour"].dt.hour.astype("int8")
model_df["dow"]        = model_df["bucket_hour"].dt.dayofweek.astype("int8")
model_df["month"]      = model_df["bucket_hour"].dt.month.astype("int8")
model_df["is_weekend"] = model_df["dow"].isin([5, 6]).astype("int8")
model_df["day_of_month"] = model_df["bucket_hour"].dt.day.astype("int8")
model_df["week_of_year"]  = model_df["bucket_hour"].dt.isocalendar().week.values.astype("int8")

# Cyclical encoding — preserves circular continuity (hour 23 ≈ hour 0)
model_df["hour_sin"]  = np.sin(2 * np.pi * model_df["hour"] / 24).astype("float32")
model_df["hour_cos"]  = np.cos(2 * np.pi * model_df["hour"] / 24).astype("float32")
model_df["dow_sin"]   = np.sin(2 * np.pi * model_df["dow"] / 7).astype("float32")
model_df["dow_cos"]   = np.cos(2 * np.pi * model_df["dow"] / 7).astype("float32")
model_df["month_sin"] = np.sin(2 * np.pi * (model_df["month"] - 1) / 12).astype("float32")
model_df["month_cos"] = np.cos(2 * np.pi * (model_df["month"] - 1) / 12).astype("float32")

# ── STEP 3: Data-driven peak-hour detection ───────────────────────────────────
print("Step 3: Peak-hour detection (learned from data)...")

# Compute average demand per hour-of-day across all AOIs
hourly_profile = demand_agg.groupby(demand_agg["bucket_hour"].dt.hour)["demand_count"].mean()
peak_threshold = hourly_profile.quantile(0.75)
peak_hours = set(hourly_profile[hourly_profile >= peak_threshold].index)
model_df["is_peak_hour"] = model_df["hour"].isin(peak_hours).astype("int8")
print(f"  Peak hours (top 25% by avg demand): {sorted(peak_hours)}")

# ── STEP 4: Interaction features ──────────────────────────────────────────────
print("Step 4: Interaction features...")

# Hour × Weekend — demand patterns shift on weekends (e.g., later morning peak)
model_df["hour_sin_x_weekend"] = (model_df["hour_sin"] * model_df["is_weekend"]).astype("float32")
model_df["hour_cos_x_weekend"] = (model_df["hour_cos"] * model_df["is_weekend"]).astype("float32")

# Peak × Weekend — weekend peaks may differ in intensity
model_df["is_peak_x_weekend"] = (model_df["is_peak_hour"] * model_df["is_weekend"]).astype("int8")

# ── STEP 5: AOI static features (spatial + busyness) ─────────────────────────
print("Step 5: AOI spatial & busyness features...")

# AOI centroids from raw delivery coordinates
aoi_centroids = (
    df.groupby("aoi_id")[["lng", "lat"]]
    .mean()
    .rename(columns={"lng": "aoi_lng", "lat": "aoi_lat"})
    .reset_index()
)
model_df = model_df.merge(aoi_centroids, on="aoi_id", how="left")
model_df["aoi_lng"] = model_df["aoi_lng"].astype("float32")
model_df["aoi_lat"] = model_df["aoi_lat"].astype("float32")

# AOI busyness — average hourly demand (captures baseline activity without using aoi_id as ordinal)
# NOTE: For strict leakage prevention, recompute on train split only in the training notebook.
aoi_busyness = (
    demand_agg.groupby("aoi_id")["demand_count"]
    .mean()
    .rename("aoi_busyness")
    .reset_index()
)
model_df = model_df.merge(aoi_busyness, on="aoi_id", how="left")
model_df["aoi_busyness"] = model_df["aoi_busyness"].fillna(0).astype("float32")
print(f"  AOIs with coordinates: {aoi_centroids['aoi_id'].nunique()}")

# ── STEP 6: Per-AOI lag, rolling, momentum, zero-demand features ─────────────
print("Step 6: Per-AOI lag/rolling/momentum/zero-demand features (this takes a while)...")

def add_aoi_features(g):
    """
    Compute all temporal features that must respect AOI boundaries.
    Input: one AOI's complete hourly series, sorted by bucket_hour.
    """
    g = g.copy()
    d = g["demand_count"]
    s = d.shift(1)  # shifted by 1 to prevent leakage from current hour

    # ── Lag features (8 lags) ──
    for lag in LAGS:
        g[f"lag_{lag}"] = d.shift(lag).astype("float32")

    # ── Rolling statistics at multiple timescales ──
    for w in ROLL_WINDOWS:
        r = s.rolling(w, min_periods=w)
        g[f"roll_{w}_mean"]  = r.mean().astype("float32")
        g[f"roll_{w}_std"]   = r.std().astype("float32")
        g[f"roll_{w}_min"]   = r.min().astype("float32")
        g[f"roll_{w}_max"]   = r.max().astype("float32")
        g[f"roll_{w}_range"] = (g[f"roll_{w}_max"] - g[f"roll_{w}_min"]).astype("float32")

    # ── Demand momentum: recent demand vs same-hour-yesterday ──
    # Captures whether demand is accelerating or decelerating
    g["momentum_1_24"] = (g["lag_1"] - g["lag_24"]).astype("float32")

    # ── Demand acceleration: change-of-change ──
    # Positive = demand growth is speeding up; negative = growth is slowing
    g["acceleration"] = (
        (g["lag_1"] - g["lag_24"]) - (g["lag_24"] - g["lag_48"])
    ).astype("float32")

    # ── Ratio features — relative demand level ──
    # lag_1 / rolling_mean: >1 means demand is above recent average (spike detection)
    roll_24 = g["roll_24_mean"].replace(0, np.nan)
    roll_168 = g["roll_168_mean"].replace(0, np.nan)
    g["lag1_over_roll24"]  = (g["lag_1"] / roll_24).fillna(1.0).astype("float32")
    g["lag1_over_roll168"] = (g["lag_1"] / roll_168).fillna(1.0).astype("float32")

    # ── Trend: short-term vs long-term baseline ──
    # >1 means recent demand is trending above the weekly average
    g["trend_24_168"] = (g["roll_24_mean"] / roll_168).fillna(1.0).astype("float32")
    g["trend_6_24"]   = (g["roll_6_mean"] / roll_24).fillna(1.0).astype("float32")

    # ── Zero-demand features ──
    # Binary: is there any demand right now?
    g["is_nonzero"] = (d > 0).astype("int8")

    # What fraction of the last 24 hours had zero demand?
    zeros_in_window = (s == 0).astype("float32")
    g["zero_ratio_24"] = zeros_in_window.rolling(24, min_periods=24).mean().astype("float32")

    # Consecutive zero hours preceding this one (dormancy streak)
    is_zero = (s == 0).astype("float32")
    cumsum_zeros = is_zero.cumsum()
    reset_at_nonzero = cumsum_zeros.where(~is_zero.astype(bool)).ffill().fillna(0)
    g["consec_zeros"] = (cumsum_zeros - reset_at_nonzero).astype("int16")

    return g

model_df = (
    model_df
    .groupby(["city", "region_id", "aoi_id"], group_keys=False)
    .apply(add_aoi_features, include_groups=False)
)
print(f"  Per-AOI features done. Shape: {model_df.shape}")

# ── STEP 7: Region-level neighbor features ────────────────────────────────────
print("Step 7: Region-level neighbor features...")

# Average lag_1 across all AOIs in the same region for each hour
# Captures: "how busy is the neighborhood right now?" — spatial context
region_lag1 = (
    model_df.groupby(["region_id", "bucket_hour"])["lag_1"]
    .mean()
    .rename("region_lag1_mean")
    .reset_index()
)
model_df = model_df.merge(region_lag1, on=["region_id", "bucket_hour"], how="left")

# Average lag_24 across region — "was the neighborhood busy at this hour yesterday?"
region_lag24 = (
    model_df.groupby(["region_id", "bucket_hour"])["lag_24"]
    .mean()
    .rename("region_lag24_mean")
    .reset_index()
)
model_df = model_df.merge(region_lag24, on=["region_id", "bucket_hour"], how="left")

# AOI demand relative to its region — captures whether this AOI is a hotspot
model_df["aoi_vs_region"] = (
    (model_df["lag_1"] / model_df["region_lag1_mean"].replace(0, np.nan))
    .fillna(1.0)
    .astype("float32")
)
print(f"  Region features merged.")

# ── STEP 8: Drop warm-up rows (NaN from lags/rolling) ────────────────────────
print("Step 8: Dropping warm-up rows...")

# The strictest requirement is roll_168_mean which needs 168 valid shifted values
# Plus shift(1) = 169 hours minimum. Drop rows where the widest window is NaN.
warmup_col = f"roll_{max(ROLL_WINDOWS)}_mean"
model_df = model_df.dropna(subset=[warmup_col]).reset_index(drop=True)
print(f"  Shape after warm-up drop: {model_df.shape}")

# Fill any remaining NaN in ratio/trend columns (edge cases where denominator was 0)
fill_cols = ["lag1_over_roll24", "lag1_over_roll168", "trend_24_168", "trend_6_24",
             "aoi_vs_region", "region_lag1_mean", "region_lag24_mean"]
for c in fill_cols:
    if c in model_df.columns:
        model_df[c] = model_df[c].fillna(0).astype("float32")

# ── STEP 9: Final column ordering & save ──────────────────────────────────────
print("Step 9: Saving...")

# Group columns logically
id_cols = ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"]
target_col = ["demand_count"]
calendar_cols = ["hour", "dow", "month", "is_weekend", "day_of_month", "week_of_year"]
cyclical_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]
peak_cols = ["is_peak_hour"]
interaction_cols = ["hour_sin_x_weekend", "hour_cos_x_weekend", "is_peak_x_weekend"]
spatial_cols = ["aoi_lng", "aoi_lat", "aoi_busyness"]
lag_cols = [f"lag_{l}" for l in LAGS]
rolling_cols = []
for w in ROLL_WINDOWS:
    rolling_cols += [f"roll_{w}_mean", f"roll_{w}_std", f"roll_{w}_min", f"roll_{w}_max", f"roll_{w}_range"]
momentum_cols = ["momentum_1_24", "acceleration", "lag1_over_roll24", "lag1_over_roll168",
                 "trend_24_168", "trend_6_24"]
zero_cols = ["is_nonzero", "zero_ratio_24", "consec_zeros"]
neighbor_cols = ["region_lag1_mean", "region_lag24_mean", "aoi_vs_region"]

all_cols = (id_cols + target_col + calendar_cols + cyclical_cols + peak_cols
            + interaction_cols + spatial_cols + lag_cols + rolling_cols
            + momentum_cols + zero_cols + neighbor_cols)

model_df = model_df[all_cols]
model_df.to_csv(OUTPUT_PATH, index=False)

print(f"\n{'='*60}")
print(f"Saved: {OUTPUT_PATH}")
print(f"Final shape: {model_df.shape}")
print(f"Total features: {len(all_cols) - len(id_cols) - len(target_col)}")
print(f"Feature groups:")
print(f"  Calendar:     {len(calendar_cols)}")
print(f"  Cyclical:     {len(cyclical_cols)}")
print(f"  Peak:         {len(peak_cols)}")
print(f"  Interaction:  {len(interaction_cols)}")
print(f"  Spatial:      {len(spatial_cols)}")
print(f"  Lag:          {len(lag_cols)}")
print(f"  Rolling:      {len(rolling_cols)}")
print(f"  Momentum:     {len(momentum_cols)}")
print(f"  Zero-demand:  {len(zero_cols)}")
print(f"  Neighbor:     {len(neighbor_cols)}")

In [ ]:
# ── Sanity Check ──────────────────────────────────────────────────────────────

preview = pd.read_csv(OUTPUT_PATH, nrows=3)
print(f"Columns ({len(preview.columns)}):")
for i, c in enumerate(preview.columns):
    print(f"  {i:2d}. {c}")

print(f"\nSample row:")
print(preview.iloc[0].to_string())

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────

n_features = len(model_df.columns) - 5 - 1  # minus id_cols minus target
print("=" * 60)
print("ADVANCED FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"  AOIs processed    : {model_df['aoi_id'].nunique()}")
print(f"  Total rows        : {len(model_df):,}")
print(f"  Total features    : {n_features}")
print(f"  Date range        : {model_df['bucket_hour'].min()} → {model_df['bucket_hour'].max()}")
print(f"  Output file       : {OUTPUT_PATH}")
print()
print("Feature breakdown:")
print(f"  Calendar + Cyclical : hour, dow, month, weekend, day_of_month, week_of_year + sin/cos encodings")
print(f"  Peak & Interaction  : data-driven peak hours, hour×weekend, peak×weekend")
print(f"  Spatial             : AOI centroid (lng/lat), busyness score")
print(f"  Lags                : {LAGS}")
print(f"  Rolling windows     : {ROLL_WINDOWS} × (mean, std, min, max, range)")
print(f"  Momentum/Trend      : momentum, acceleration, demand ratios, short/long trend")
print(f"  Zero-demand         : is_nonzero, zero_ratio_24, consecutive zeros")
print(f"  Neighbor/Region     : region_lag1_mean, region_lag24_mean, aoi_vs_region")
print()
print("NOTE: aoi_id and region_id are kept as ID columns only — NOT used as numeric features.")
print("      Use target encoding or embeddings in the training notebook if needed.")

In [15]:
aoi_totals = demand_agg.groupby("aoi_id")["demand_count"].sum().sort_values(ascending=False)
coverage = aoi_totals.cumsum() / aoi_totals.sum()

n_70 = (coverage <= 0.70).sum()
n_80 = (coverage <= 0.80).sum()
n_90 = (coverage <= 0.90).sum()
n_100 = (coverage <= 1.00).sum()


print(f"AOIs needed for 70% demand coverage: {n_70}")
print(f"AOIs needed for 80% demand coverage: {n_80}")
print(f"AOIs needed for 90% demand coverage: {n_90}")
print(f"AOIs needed for 100% demand coverage: {n_100}")


AOIs needed for 70% demand coverage: 1353
AOIs needed for 80% demand coverage: 2214
AOIs needed for 90% demand coverage: 4039
AOIs needed for 100% demand coverage: 17714
